# Cellfix5 — full39 sau human admission

Probe đã audit: 20261017T173345Z-905edcb5. Human reviewer: Trí Công.
Review mới đủ 18 ca/90 tiêu chí đã qua require_probe_review local.
Notebook nhúng source cellfix5, ZIP probe gốc và JSON review đúng hash.
Không cần chạy lại smoke/probe hoặc upload thêm file.

Dùng runtime mới, GPU T4, Run all bốn code cell:
1. Kiểm payload/hash; giữ ZIP gốc và tạo reviewed-probe riêng.
2. Dựng môi trường cô lập theo dependency/CUDA pins đã chạy thành công.
3. Admission lại bằng runner trước model-cache; 181 regression, CUDA và ba validate-only.
4. Chạy full39; tải ZIP kể cả khi trượt gate hoặc timeout.

Giữ nguyên Qwen/Qwen2.5-3B-Instruct revision aa8e72537993ba99e69dfaafa59ed015b17504d1,
NF4/float16/seed42, 8 calls/ca, 2 attempts/stage, 1536 tokens/call, 180s/ca;
tối đa 312 calls, 479232 generated tokens, worker 5400s, notebook watchdog 5520s.
Đạt tự động cần 39/39 nhãn đúng, false_support=0, runtime_failed_rows=0 và mọi check đạt.

Full39 của candidate chưa chạy tại thời điểm đóng gói. Sau khi gửi ZIP còn phải audit/replay
và review evidence, population/coverage, numeric binding, missing information từng ca.
18/39 là development data; chưa chứng minh generalization hoặc E2E Reader/Summarizer.
Giữ ZIP tải cuối cùng, kể cả có thêm (1) trong tên, rồi dừng GPU.


**1 — Candidate và review**

In [ ]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
import base64, hashlib, io, json, os, signal, subprocess, sys, time, uuid, zipfile
from google.colab import files

smoke_passed = False
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'-'+uuid.uuid4().hex[:8]
run_root = Path('/content')/('atomic-cellfix5-full39-'+run_id)
root = run_root/'workspace'
out = run_root/('output-cellfix5-full39-'+run_id)
root.mkdir(parents=True,exist_ok=False)
out.mkdir(exist_ok=False)
statuses = {}
env = os.environ.copy()
env.update(PYTHONPATH=str(root),HF_HOME=str(run_root/'hf'),
    HF_HUB_DISABLE_PROGRESS_BARS='1',TOKENIZERS_PARALLELISM='false')
env.pop('HF_HUB_OFFLINE',None)
env.pop('TRANSFORMERS_OFFLINE',None)
venv = run_root/'venv'
python = str(venv/'bin/python')
payload = '...'
payload_sha256 = '922bb57ed569982785d026e365c208ef19d4fc5c104e622aa7e9e70395fcb869'
manifest = {'04_factchecker/checkpoints/factchecker_merged/chat_template.jinja': {'size': 2507, 'sha256': 'cd8e9439f0570856fd70470bf8889ebd8b5d1107207f67a5efb46e342330527f'}, '04_factchecker/checkpoints/factchecker_merged/config.json': {'size': 1548, 'sha256': '2e13d6edf2413d55422a2fe541d4fc851e17b01cc7a16601a107547687e7951d'}, '04_factchecker/checkpoints/factchecker_merged/generation_config.json': {'size': 242, 'sha256': 'dd1a52c6b9e837741be9dfa5c7fb8f0d8d9ed57e06d399a0fe52ec23d10ca066'}, '04_factchecker/checkpoints/factchecker_merged/tokenizer.json': {'size': 11421892, 'sha256': '3fd169731d2cbde95e10bf356d66d5997fd885dd8dbb6fb4684da3f23b2585d8'}, '04_factchecker/checkpoints/factchecker_merged/tokenizer_config.json': {'size': 664, 'sha256': 'd9bf1d78a80bad5eb65b83c2608910e3be00a9d3250c8153a5a393ad1257e1d6'}, '04_factchecker/config.py': {'size': 2026, 'sha256': '2d9f99a6dee5a7a4b40278a552002d7d9737e060730bc9d5e48de4923693a584'}, '04_factchecker/eval_factchecker.py': {'size': 7759, 'sha256': '579907c751d107f9b1634511501b9dfd433ac9d583d551d989e1b95f6f7cbc0e'}, '04_factchecker/generate_training_data.py': {'size': 6255, 'sha256': 'e42e8abde6571f2257b2a05500b73395ec51c41407e4b35b872ef38684533e38'}, '04_factchecker/inference/__init__.py': {'size': 474, 'sha256': '11edee781c7d9592646acc9ebbca280c818fe561abb487995b079cbbe3ac6495'}, '04_factchecker/inference/action_recommender.py': {'size': 1846, 'sha256': '9e8d9350142122e573ed4ab758581b0076dc39d339566dd598c9f757fba9c37d'}, '04_factchecker/inference/atomic_decoding.py': {'size': 19686, 'sha256': '12b54889e184ef9fa5f5cb9300fe3150560fd2fc7f41d2519ae95873b178b31d'}, '04_factchecker/inference/atomic_verifier.py': {'size': 68579, 'sha256': '35cec201bec6d1a4ab5d6186a7a0617c288db92ef7bd54d69e679bf2a6e3f595'}, '04_factchecker/inference/confidence.py': {'size': 3463, 'sha256': '307cdfb8bb9509a5189a31994d0aaa4c37f662e008f36dadaa54dcabeba0a09b'}, '04_factchecker/inference/evidence_screen.py': {'size': 5405, 'sha256': '23574b7320ac576989b283a58d3e81d58be001f1b1c286523d4f0baa04dfcab5'}, '04_factchecker/inference/judge.py': {'size': 5412, 'sha256': '818aa5a4dc7f8065e101264e59d57828cc00f60d1c105156e49c03f374f6dbb7'}, '04_factchecker/inference/numeric_verifier.py': {'size': 14642, 'sha256': '13facfbf47e4a979ea29c37ac81ae65aab866a0e30905b3a4b484c40c0df5428'}, '04_factchecker/inference/verifier.py': {'size': 11971, 'sha256': '7eff9f8678cf1a4c270e59ada432bdf12a2fd6323c3d598fa34cc3513fd468f8'}, '04_factchecker/merge_lora_weights.py': {'size': 6730, 'sha256': 'bb237ec90020328f9cb2a849f75f4688e913bc41f35b661781ad113a4af78420'}, '04_factchecker/node.py': {'size': 7793, 'sha256': '577e62d2ce330d19bc4791641066d8d17b8dc4baebfc3dab455d593658afc48c'}, '04_factchecker/prompts.py': {'size': 7967, 'sha256': 'fe7be89a4ac6f789f9629f90c9710aeafc074806c066fa2451cb889121b6017e'}, '04_factchecker/train_factchecker_qlora.py': {'size': 21854, 'sha256': '99cee60ec35a3b60f9da41719891c3130ec5c64bf38e7e5bd4872382b13cc379'}, '04_factchecker/validate_labels.py': {'size': 3825, 'sha256': '1c70278ad4ab59659dade53b8f8eb6f4a1cf135d7ab957d685b248a5b1cd9877'}, 'requirements-atomic-verifier.txt': {'size': 131, 'sha256': 'b95eaa539ed6046f9261b09cdaa82017360568703d8af8d2bc6d8eed3c9af370'}, 'requirements-orchestrator.txt': {'size': 467, 'sha256': '31b5dac851eaaf005a151e9c071226aeafeb4679ebe4019d0368e68fbd0dacd4'}, 'research_rag/__init__.py': {'size': 134, 'sha256': 'cc7fea0e7e9fe57abaea1ca3e78e70ecaad9feb9819b088c50cdf26ba0d249e5'}, 'research_rag/adapters.py': {'size': 5305, 'sha256': '22537d4284338c66a1262d03dd3ce55dea4a95a2fa859ae78c638b66c45f147c'}, 'research_rag/controller.py': {'size': 7426, 'sha256': 'b97466aece81f27bd89d62f1fd0c61857a8a604b2b9681faf1850cdfb55ff43f'}, 'research_rag/evaluate.py': {'size': 10760, 'sha256': '48f5936f8ef3e3454ab7d2049d803be18d67134279fe14359e4c2ec0451fcf66'}, 'research_rag/evaluation/__init__.py': {'size': 66, 'sha256': 'edf15e92c3397d31441733113a0ea6bc45d805a6084ef354526ffce327d7dd5d'}, 'research_rag/evaluation/atomic_verifier_contract_v8.json': {'size': 8468, 'sha256': 'd271a605783c4bd294e500c1488e56974d3b0759edc501b9e725515cf9a2edd6'}, 'research_rag/evaluation/atomic_verifier_contract_v8_cellfix1.json': {'size': 8995, 'sha256': '84e42bb71fe0d088adf3abe3ebd5660d3a2cdfe4f0d601e943cdaa04706c4cf5'}, 'research_rag/evaluation/atomic_verifier_contract_v8_cellfix2.json': {'size': 9302, 'sha256': '086fb6f181361ee64885c915b1cc67f636870b3b264a8017245fd566edf74574'}, 'research_rag/evaluation/atomic_verifier_contract_v8_cellfix3.json': {'size': 9978, 'sha256': '82feadb1b0cfb15490a825867ec3a2159cfa5253cc2fa79267ed9525907cf126'}, 'research_rag/evaluation/atomic_verifier_contract_v8_cellfix4.json': {'size': 10847, 'sha256': '80f1c75d8e91230a1d72a495f49b8dfcfa9bde57c2632b76d1a381624fdb6322'}, 'research_rag/evaluation/atomic_verifier_contract_v8_cellfix5.json': {'size': 12243, 'sha256': 'c37fda6dded8c7faa34dc7dbad4523944e07c92d20741832dc72069e0b7d0057'}, 'research_rag/evaluation/atomic_verifier_criteria_v8.json': {'size': 12958, 'sha256': '642d66612c25514e3764840f750a97cc85c93d4a32a3e4f23bcabff6b993e84c'}, 'research_rag/evaluation/atomic_verifier_logic_v3_baseline.json': {'size': 31343, 'sha256': '42f8fab1e7b5a9056538481113d6590d4e84d0b089989472afe09f9ccd16be85'}, 'research_rag/evaluation/blind_verifier_scope_regression_cases.jsonl': {'size': 16356, 'sha256': 'f349b1d6a50ca417c28c43d5fced1f064fc43ebdf362d70acfc574d7d37ed45c'}, 'research_rag/evaluation/compare_blind_verifier.py': {'size': 16716, 'sha256': 'ea14524f1f43c806975e0adbe0c0f24eecf23fd970123be735b9e14b7e6644cb'}, 'research_rag/evaluation/compare_confidence_backends.py': {'size': 10701, 'sha256': '0d383a92d02cec05743c3585058cd1fd7a4dbd5daab3730beee08b499eb5e873'}, 'research_rag/evaluation/compare_factchecker.py': {'size': 18763, 'sha256': 'ce7a74e56699a35d597493dd5152972a0cf40447aab6000774c7d76209574ae7'}, 'research_rag/evaluation/decomposition_probe_claims_v2.jsonl': {'size': 1711, 'sha256': '440030681bca80724ab4ee0e85e79ac70db7f1703f77587991000cb07b6bea13'}, 'research_rag/evaluation/evaluate_atomic_verifier.py': {'size': 33207, 'sha256': 'f8e3362b38c4e1cf11b2d3d0d90f10d7abdf8647e99f3cad5d6a8080a393f9ed'}, 'research_rag/evaluation/factchecker_contract_cases.jsonl': {'size': 3533, 'sha256': '918a1a95cfed421aa8748d7eac29af74a03f93825c30817e002b2b2e16feb7e3'}, 'research_rag/graph.py': {'size': 32194, 'sha256': 'b2883b3b7c2ca75cda20ab9223fb8012591dcd3284dc2eca9595af083fc275df'}, 'research_rag/indexing.py': {'size': 5245, 'sha256': '2e02675e5094a278b4eb9d0e2eb0245389fda47ad0d9236ba69fef68f457cd14'}, 'research_rag/local.py': {'size': 17378, 'sha256': 'b363825e64d07a5f8c12c5b456b1e58e4b2c4297e93f48d493c5dec3ba8bbf5a'}, 'research_rag/observe.py': {'size': 2854, 'sha256': '46300c0d7f859228523efc2359d01092c48d7317b7509f5f4efb8fc6bbaae051'}, 'research_rag/planning.py': {'size': 9967, 'sha256': '680b25c3d049f7fa908ebaaed9b2a799a2430425eefd33ffb2b362e36a2d24f7'}, 'research_rag/preflight.py': {'size': 5027, 'sha256': 'bccd1755d5e23462f334703822d6fb185ca089bd62e2be9214bc0b9558fb5e39'}, 'research_rag/retrieval.py': {'size': 12796, 'sha256': 'f8fc0c3118c0feb7dcb845f6be2a3dc026f253679524cbdcb5c4ec9bc1f6c75b'}, 'research_rag/state.py': {'size': 1901, 'sha256': '0da55a10e4f5d1c33b4f06cad2e78553bfb190bc30c8b96b76a699e69a6fdc39'}, 'research_rag/tests/fixtures/atomic_cellfix2_binding_traces.json': {'size': 19180, 'sha256': '734d96996de0008940e908d9513b5e21dd331d64c6124209c83e0af651cdbf9f'}, 'research_rag/tests/fixtures/atomic_cellfix3_citation_traces.json': {'size': 4284, 'sha256': '62ea69491c48c58c7441023f8314b04b9d0ef63b88fbcf58504654d94b891d82'}, 'research_rag/tests/fixtures/atomic_cellfix4_metric_traces.json': {'size': 1651, 'sha256': '19f7e6de615f35b5a5bbf276aa0639930b9c35140cd9ba9b2ab2a0a92a28187d'}, 'research_rag/tests/fixtures/atomic_v8_v7_assessments.json': {'size': 4591, 'sha256': '577eb9dd475a800b80b8330510573c0b0c52b53e57e565c77af79e7b71675494'}, 'research_rag/tests/test_atomic_assessment_v8.py': {'size': 16879, 'sha256': '93e900e49a79b22313d0c74c5b3d1e8ff492292a7ad2674f87b12fcf7dcd9db5'}, 'research_rag/tests/test_atomic_contract_v5.py': {'size': 10515, 'sha256': 'c4bec92fda42241f7c0ee03366dda8e4ed88049e1ac5b75a0780f9f910c0cbed'}, 'research_rag/tests/test_atomic_decoding.py': {'size': 8267, 'sha256': '6eabd3db1e06ffa61021811b7909b65277e6d4719cbf7d1f3e26d74c79e1745b'}, 'research_rag/tests/test_atomic_decomposition_v5.py': {'size': 8424, 'sha256': 'f5f2e053f0dbe3de38e0751aded20f22bdfa7c3822e4e7b669dd3eea2baf82d9'}, 'research_rag/tests/test_atomic_experiment.py': {'size': 18569, 'sha256': '9f284c20054211fa851a786b96424ca531f9112383a918a2fbee09b0fd605eaa'}, 'research_rag/tests/test_atomic_metric_cellfix5.py': {'size': 13444, 'sha256': 'fc66c70e90689eb9816ba8a8a4c6fe028832520e9bde12993ff97d3ad0f994fb'}, 'research_rag/tests/test_atomic_metric_v6.py': {'size': 7270, 'sha256': '63433a23b9b07a309c2f88afd8bbe6eb6c552a344d69a9e97a25dedbee095477'}, 'research_rag/tests/test_atomic_metric_v7.py': {'size': 15499, 'sha256': '6d05e7f7502457cec1962b0959c731e13b123e678fee19c0fe156423eab29299'}, 'research_rag/tests/test_atomic_smoke_cellfix.py': {'size': 5569, 'sha256': '9b4efc67525f4fc905739590839fcb54c9db5b0c55964a429469b86b0a428949'}, 'research_rag/tests/test_atomic_smoke_cellfix2.py': {'size': 3574, 'sha256': '6c99a4ed08116f7bb5a72790a6830bcf4417fdf8168a404bc35fc51061c3f11a'}, 'research_rag/tests/test_atomic_smoke_cellfix3.py': {'size': 12622, 'sha256': 'f240bd059d220ba21cca5a02deb17e39302cec73c22c1a473825599a44002d7b'}, 'research_rag/tests/test_atomic_smoke_cellfix4.py': {'size': 9804, 'sha256': '51009a2b500059634383a6a1961d9caf8d9156762f06fb9524b705562d728c49'}, 'research_rag/tests/test_atomic_verifier.py': {'size': 25714, 'sha256': '5d039eddaed6f8cd4a8a1e6677c52d012d7648d20b3437c042c0a0097e0491bb'}, 'research_rag/tests/test_summary_schema.py': {'size': 2852, 'sha256': '95f04375510bca135ab56a31c40b1e6b418e126d0b5f9059db8da60dd71b521c'}, 'research_rag/tests/test_summary_trace.py': {'size': 4974, 'sha256': '5285b7109213573ef157e3b8d67e44d6eb78bc9e8b7e3d18290e7cbad5c47538'}, 'train_agent3_summarizer/config.py': {'size': 5213, 'sha256': 'cf69370bc520b960cf144386793f44fac316c1c1ea20fd9bb3e566519c6863ea'}, 'train_agent3_summarizer/critic/__init__.py': {'size': 0, 'sha256': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855'}, 'train_agent3_summarizer/critic/prompts.py': {'size': 3000, 'sha256': '5ec5559e8c6542375feb0d6ba619dc8b64036ae748972f9a01b8dfcb3ea54c11'}, 'train_agent3_summarizer/critic/train_critic_qlora.py': {'size': 19528, 'sha256': 'abf7cdf00ad5c1c76814e2a9425cb19088690a35ccfad0593b31d2d0f5e123d2'}, 'train_agent3_summarizer/eval_generator_critic.py': {'size': 27382, 'sha256': '9f0867034b0ac472a89afd0d8a6aa41b3a00e13ba89c2defbb7376c985f59e2b'}, 'train_agent3_summarizer/generate_training_data.py': {'size': 18476, 'sha256': 'ac2ec01492e5a73bf99700b8f189d247b451d42203c1447397ac8075f9f024ae'}, 'train_agent3_summarizer/generator/__init__.py': {'size': 0, 'sha256': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855'}, 'train_agent3_summarizer/generator/prompts.py': {'size': 1944, 'sha256': '64bf3da00181c210452f86f24cf67b9c8acd9ea12c66e5d33eb2f026327e46fd'}, 'train_agent3_summarizer/generator/train_generator_qlora.py': {'size': 19436, 'sha256': '1262f59c2a3a6be7815f9c3dee434bdde047a5e8a6da9ade82c84a06c9c34d11'}, 'train_agent3_summarizer/llm_judge_eval.py': {'size': 6421, 'sha256': 'cc1cfc18b9f1135e54fe35dd3702ad1724d1e7ca5ebc4b4e98b8273edd37f413'}, 'train_agent3_summarizer/merge_lora_weights.py': {'size': 2558, 'sha256': '65dcb7ed6f2711c01fe9a98b2bf83225d22aacaf11219d7fc8bd92f8ca33af8b'}, 'train_agent3_summarizer/node.py': {'size': 15263, 'sha256': '43dff6e0ecccd9b9f7c6eb24d1ee73c0239c45a0c3553faf499a5e9fcfad5096'}, 'train_agent3_summarizer/validate_evidence_span.py': {'size': 5874, 'sha256': '54795f2bf992d07a32a70c8f19852d022f1a4706dbb0b180cf87840bb43f17bf'}}
criteria_sha256 = '642d66612c25514e3764840f750a97cc85c93d4a32a3e4f23bcabff6b993e84c'
def bundle():
    # This dedicated directory contains only diagnostics produced by these cells.
    entries = [{'path':str(p.relative_to(out)), 'size':p.stat().st_size,
                'sha256':hashlib.sha256(p.read_bytes()).hexdigest()}
               for p in sorted(out.rglob('*')) if p.is_file() and p.name != 'output-manifest.json']
    (out/'output-manifest.json').write_text(json.dumps({'run_id':run_id,'files':entries},indent=2))
    archive = out.with_suffix('.zip')
    pending = archive.with_suffix('.zip.partial')
    with zipfile.ZipFile(pending, 'w', zipfile.ZIP_DEFLATED) as z:
        for p in sorted(out.rglob('*')):
            if p.is_file(): z.write(p, str(p.relative_to(out)))
    pending.replace(archive)
    print('Diagnostic archive:', archive.name, flush=True)

def run(name, command, *, timeout=1800, child_env=None, required=True):
    assert name not in statuses, 'Preserve the previous stage; start a fresh run for retries'
    started = time.perf_counter()
    status = {'state':'running'}
    statuses[name] = status
    (out/'status.json').write_text(json.dumps(statuses,indent=2))
    try:
        with (out/(name+'.stdout.log')).open('x') as stdout, (out/(name+'.stderr.log')).open('x') as stderr:
            process = subprocess.Popen(command, cwd=root, env=child_env or env,
                                       stdout=stdout, stderr=stderr, start_new_session=True)
            try:
                while True:
                    try:
                        code = process.wait(timeout=15)
                        break
                    except subprocess.TimeoutExpired:
                        elapsed = time.perf_counter()-started
                        print(name, 'running', round(elapsed), 'seconds', flush=True)
                        if elapsed > timeout: raise TimeoutError(name)
            except BaseException:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    os.killpg(process.pid, signal.SIGKILL)
                    process.wait()
                raise
        status.update(state='completed' if code == 0 else 'failed', exit_code=code)
    except BaseException as exc:
        status.update(state='interrupted', error_type=type(exc).__name__)
        raise
    finally:
        status['elapsed_ms'] = round((time.perf_counter()-started)*1000,2)
        (out/'status.json').write_text(json.dumps(statuses,indent=2))
        bundle()
    print(name, status, flush=True)
    if required and code:
        print((out / (name + '.stderr.log')).read_text()[-12000:], flush=True)
        raise RuntimeError(name + ' failed; inspect its saved stderr')
    return code

def download_archive():
    bundle()
    files.download(str(out.with_suffix('.zip')))

try:
    raw=base64.b64decode(payload,validate=True)
    assert hashlib.sha256(raw).hexdigest()==payload_sha256
    with zipfile.ZipFile(io.BytesIO(raw)) as archive:
        assert archive.testzip() is None
        assert len(archive.namelist())==len(manifest) and set(archive.namelist())==set(manifest)
        for name,item in manifest.items():
            relative=PurePosixPath(name)
            assert not relative.is_absolute() and '..' not in relative.parts and '\\' not in name
            data=archive.read(name)
            assert len(data)==item['size'] and hashlib.sha256(data).hexdigest()==item['sha256'],name
            target=root/relative
            target.parent.mkdir(parents=True,exist_ok=True)
            target.write_bytes(data)
    assert hashlib.sha256((root/'research_rag/evaluation/atomic_verifier_criteria_v8.json').read_bytes()).hexdigest()==criteria_sha256
    (out/'input-manifest.json').write_text(json.dumps(manifest,indent=2))
    (out/'frozen-inputs.zip').write_bytes(raw)
    (out/'notebook-provenance.json').write_text(json.dumps({
        'candidate':'atomic-verifier-constrained-v8-cellfix5',
        'criteria_sha256':criteria_sha256,'payload_sha256':payload_sha256,
        'startup':'fresh embedded inputs; no previous session required',
        'model_data_budgets_and_gates_unchanged':True},indent=2))
    bundle()
except BaseException:
    download_archive()
    raise
print('Frozen cellfix5 ready. Run directory:',run_root)

import shutil
probe_zip_b64 = '...'
probe_zip_sha256 = 'e854d7d7276a806941b876ef1a23bc04365677caa2301cda3c9f14356dc4f1bb'
review_b64 = 'ewogICJwcmVkaWN0aW9uc19zaGEyNTYiOiAiYjU1NTU1MWI3ODEyZmQ5YWE5ZmNmMzcyNzE1ZjE1NTFjY2Q3MmQxNTY4MTc2ZGU1MDI2NmMxMjEwYzkwYjAxNCIsCiAgImNyaXRlcmlhX3NoYTI1NiI6ICI2NDJkNjY2MTJjMjU1MTRlMzc2NDg0MGY3NTBhOTdjYzg1YzkzZDRhMzJhM2U0ZjIzYmNhYmZmNmI5OTNlODRjIiwKICAicmV2aWV3ZXIiOiAiVHLDrSBDw7RuZyIsCiAgInJldmlld2VyX2tpbmQiOiAiaHVtYW4iLAogICJyZXZpZXdfY29tcGxldGVkIjogdHJ1ZSwKICAiY2FzZXMiOiBbCiAgICB7CiAgICAgICJleGFtcGxlX2lkIjogImRlY29tcG9zZTowMDEiLAogICAgICAiY2xhaW0iOiAiTW9kZWwgSnVuaXBlciB1c2VzIDI1JSBsZXNzIG1lbW9yeSBhbmQgcmV0YWlucyA5NCUgYWNjdXJhY3kuIiwKICAgICAgImp1ZGdtZW50cyI6IHsKICAgICAgICAiZmFjdHVhbF9wcmVkaWNhdGVzX25vdF9mcmFnbWVudHMiOiB0cnVlLAogICAgICAgICJhbGxfcmVxdWlyZW1lbnRzX3ByZXNlcnZlZCI6IHRydWUsCiAgICAgICAgInF1YW50aWZpZXJfcHJlc2VydmVkIjogdHJ1ZSwKICAgICAgICAiZW50aXR5X21ldHJpY19iaW5kaW5nX2NvcnJlY3QiOiB0cnVlLAogICAgICAgICJ0aW1lX2NvbmRpdGlvbl9zY29wZV9jb3JyZWN0IjogdHJ1ZQogICAgICB9LAogICAgICAibm90ZXMiOiAiSGFpIHByZWRpY2F0ZSB1c2VzIDI1JSBsZXNzIG1lbW9yeSB2w6AgcmV0YWlucyA5NCUgYWNjdXJhY3kgxJHGsOG7o2MgdMOhY2ggxJHDum5nOyBwYXJ0IHNhdSBr4bq/IHRo4burYSBNb2RlbCBKdW5pcGVyLiBIYWkgbWV0cmljIG1lbW9yeS9hY2N1cmFjeSB2w6AgaGFpIHJlcXVpcmVtZW50IDI1JS85NCUga2jDtG5nIHRy4buZbiBuaGF1OyBxdWFuIGjhu4cgbGVzcy9yZXRhaW5zIHbhuqtuIG7hurFtIHRyb25nIHF1b3RlLiIKICAgIH0sCiAgICB7CiAgICAgICJleGFtcGxlX2lkIjogImRlY29tcG9zZTowMDIiLAogICAgICAiY2xhaW0iOiAiTW9kZWwgSnVuaXBlciByZXRhaW5zIDk0JSBhY2N1cmFjeS4iLAogICAgICAianVkZ21lbnRzIjogewogICAgICAgICJmYWN0dWFsX3ByZWRpY2F0ZXNfbm90X2ZyYWdtZW50cyI6IHRydWUsCiAgICAgICAgImFsbF9yZXF1aXJlbWVudHNfcHJlc2VydmVkIjogdHJ1ZSwKICAgICAgICAicXVhbnRpZmllcl9wcmVzZXJ2ZWQiOiB0cnVlLAogICAgICAgICJlbnRpdHlfbWV0cmljX2JpbmRpbmdfY29ycmVjdCI6IHRydWUsCiAgICAgICAgInRpbWVfY29uZGl0aW9uX3Njb3BlX2NvcnJlY3QiOiB0cnVlCiAgICAgIH0sCiAgICAgICJub3RlcyI6ICJN4buZdCBwcmVkaWNhdGUgZ2nhu68gbmd1ecOqbiByZXRhaW5zIDk0JSBhY2N1cmFjeTsgZW50aXR5IE1vZGVsIEp1bmlwZXIsIG1ldHJpYyBhY2N1cmFjeSwgcmVxdWlyZW1lbnQgOTQlLiBLaMO0bmcgdOG7sSB0aMOqbSBwb3B1bGF0aW9uIGhv4bq3YyByZXN0cmljdGlvbi4iCiAgICB9LAogICAgewogICAgICAiZXhhbXBsZV9pZCI6ICJkZWNvbXBvc2U6MDAzIiwKICAgICAgImNsYWltIjogIlRoZSBzdHVkeSB1c2VkIGh1bWFuIGxhYmVscy4iLAogICAgICAianVkZ21lbnRzIjogewogICAgICAgICJmYWN0dWFsX3ByZWRpY2F0ZXNfbm90X2ZyYWdtZW50cyI6IHRydWUsCiAgICAgICAgImFsbF9yZXF1aXJlbWVudHNfcHJlc2VydmVkIjogdHJ1ZSwKICAgICAgICAicXVhbnRpZmllcl9wcmVzZXJ2ZWQiOiB0cnVlLAogICAgICAgICJlbnRpdHlfbWV0cmljX2JpbmRpbmdfY29ycmVjdCI6IHRydWUsCiAgICAgICAgInRpbWVfY29uZGl0aW9uX3Njb3BlX2NvcnJlY3QiOiB0cnVlCiAgICAgIH0sCiAgICAgICJub3RlcyI6ICJN4buZdCBwcmVkaWNhdGUgdXNlZCBodW1hbiBsYWJlbHMgxJHhuqd5IMSR4bunIGPhuqMgaMOgbmggxJHhu5luZyB2w6AgxJHhu5FpIHTGsOG7o25nLiBNZXRyaWMgdXNlZCBodW1hbiBsYWJlbHMgaOG7o3AgbOG7hyB2w6wgc291cmNlIGtow7RuZyBuw6p1IG3hu5l0IG5hbWVkIG1lYXN1cmVtZW50IHJpw6puZy4iCiAgICB9LAogICAgewogICAgICAiZXhhbXBsZV9pZCI6ICJkZWNvbXBvc2U6MDA0IiwKICAgICAgImNsYWltIjogIlRoZSBuZXcgaW5kZXggY2F1c2VkIHRoZSBsYXRlbmN5IGltcHJvdmVtZW50LiIsCiAgICAgICJqdWRnbWVudHMiOiB7CiAgICAgICAgImZhY3R1YWxfcHJlZGljYXRlc19ub3RfZnJhZ21lbnRzIjogdHJ1ZSwKICAgICAgICAiYWxsX3JlcXVpcmVtZW50c19wcmVzZXJ2ZWQiOiB0cnVlLAogICAgICAgICJxdWFudGlmaWVyX3ByZXNlcnZlZCI6IHRydWUsCiAgICAgICAgImVudGl0eV9tZXRyaWNfYmluZGluZ19jb3JyZWN0IjogdHJ1ZSwKICAgICAgICAidGltZV9jb25kaXRpb25fc2NvcGVfY29ycmVjdCI6IHRydWUKICAgICAgfSwKICAgICAgIm5vdGVzIjogIkdp4buvIG5ndXnDqm4gY2F1c2FsIHByZWRpY2F0ZSBjYXVzZWQgdGhlIGxhdGVuY3kgaW1wcm92ZW1lbnQgdHJvbmcgcXVvdGUsIGtow7RuZyBiaeG6v24gY2F1c2F0aW9uIHRow6BuaCBjb3JyZWxhdGlvbi4gRW50aXR5IG5ldyBpbmRleCB2w6AgbWV0cmljIGxhdGVuY3kgxJHDum5nOyBhc3Nlc3NtZW50IHbhuqtuIHBo4bqjaSBraeG7g20gcXVhbiBo4buHIGNhdXNlZCB04burIHRvw6BuIGLhu5kgcXVvdGUuIgogICAgfSwKICAgIHsKICAgICAgImV4YW1wbGVfaWQiOiAiZGVjb21wb3NlOjAwNSIsCiAgICAgICJjbGFpbSI6ICJBbGwgdGFza3MgaW1wcm92ZWQgYWZ0ZXIgY29tcHJlc3Npb24uIiwKICAgICAgImp1ZGdtZW50cyI6IHsKICAgICAgICAiZmFjdHVhbF9wcmVkaWNhdGVzX25vdF9mcmFnbWVudHMiOiB0cnVlLAogICAgICAgICJhbGxfcmVxdWlyZW1lbnRzX3ByZXNlcnZlZCI6IHRydWUsCiAgICAgICAgInF1YW50aWZpZXJfcHJlc2VydmVkIjogdHJ1ZSwKICAgICAgICAiZW50aXR5X21ldHJpY19iaW5kaW5nX2NvcnJlY3QiOiB0cnVlLAogICAgICAgICJ0aW1lX2NvbmRpdGlvbl9zY29wZV9jb3JyZWN0IjogdHJ1ZQogICAgICB9LAogICAgICAibm90ZXMiOiAiQWxsIMSRxrDhu6NjIMOhbmggeOG6oSBhbGwgdHLDqm4gcG9wdWxhdGlvbiB0YXNrczsgaW1wcm92ZWQgbMOgIHByZWRpY2F0ZSBkdXkgbmjhuqV0IHNvdXJjZSBjdW5nIGPhuqVwLCBraMO0bmcgdOG7sSDEkW/DoW4gYWNjdXJhY3kvbGF0ZW5jeS4gYWZ0ZXIgY29tcHJlc3Npb24gZ2nhu68gxJHDum5nIHRo4budaSBnaWFuLiIKICAgIH0sCiAgICB7CiAgICAgICJleGFtcGxlX2lkIjogImRlY29tcG9zZTowMDYiLAogICAgICAiY2xhaW0iOiAiVGhlIHJlcG9ydCBldmFsdWF0ZWQgZm91ciBtb2RlbHMgb24gdHdvIGRhdGFzZXRzLiIsCiAgICAgICJqdWRnbWVudHMiOiB7CiAgICAgICAgImZhY3R1YWxfcHJlZGljYXRlc19ub3RfZnJhZ21lbnRzIjogdHJ1ZSwKICAgICAgICAiYWxsX3JlcXVpcmVtZW50c19wcmVzZXJ2ZWQiOiB0cnVlLAogICAgICAgICJxdWFudGlmaWVyX3ByZXNlcnZlZCI6IHRydWUsCiAgICAgICAgImVudGl0eV9tZXRyaWNfYmluZGluZ19jb3JyZWN0IjogdHJ1ZSwKICAgICAgICAidGltZV9jb25kaXRpb25fc2NvcGVfY29ycmVjdCI6IHRydWUKICAgICAgfSwKICAgICAgIm5vdGVzIjogIkdp4buvIGV2YWx1YXRlZCBmb3VyIG1vZGVscyB2w6AgY29uZGl0aW9uIG9uIHR3byBkYXRhc2V0cy4gTnVtZXJpYyByZXF1aXJlbWVudHMgZ+G7k20gNCBtb2RlbHMgdsOgIDIgZGF0YXNldHMsIG3hu5dpIHPhu5EgY8OzIG9iamVjdCByacOqbmc7IGtow7RuZyBjaOG7iSBraeG7g20gc+G7kSA0LiIKICAgIH0sCiAgICB7CiAgICAgICJleGFtcGxlX2lkIjogImRlY29tcG9zZTowMDciLAogICAgICAiY2xhaW0iOiAiVHJhaW5pbmcgdG9vayAxMiBzZWNvbmRzLiIsCiAgICAgICJqdWRnbWVudHMiOiB7CiAgICAgICAgImZhY3R1YWxfcHJlZGljYXRlc19ub3RfZnJhZ21lbnRzIjogdHJ1ZSwKICAgICAgICAiYWxsX3JlcXVpcmVtZW50c19wcmVzZXJ2ZWQiOiB0cnVlLAogICAgICAgICJxdWFudGlmaWVyX3ByZXNlcnZlZCI6IHRydWUsCiAgICAgICAgImVudGl0eV9tZXRyaWNfYmluZGluZ19jb3JyZWN0IjogdHJ1ZSwKICAgICAgICAidGltZV9jb25kaXRpb25fc2NvcGVfY29ycmVjdCI6IHRydWUKICAgICAgfSwKICAgICAgIm5vdGVzIjogIk1ldHJpYyB0b29rIDEyIHNlY29uZHMgZ2nhu68gcXVhbiBo4buHIHRo4budaSBsxrDhu6NuZyB2w6AgcXVhbnRpdHksIGtow7RuZyBjw7JuIHRvb2sgxJHGoW4gbOG6uyBob+G6t2MgMTIgc2Vjb25kcyB0aGnhur91IHF1YW4gaOG7hy4gRW50aXR5IFRyYWluaW5nLCB1bml0IHNlY29uZHMgxJHDum5nLiIKICAgIH0sCiAgICB7CiAgICAgICJleGFtcGxlX2lkIjogImRlY29tcG9zZTowMDgiLAogICAgICAiY2xhaW0iOiAiVGhlIHNhbXBsZSBjb250YWluZWQgMjQwIGRvY3VtZW50cy4iLAogICAgICAianVkZ21lbnRzIjogewogICAgICAgICJmYWN0dWFsX3ByZWRpY2F0ZXNfbm90X2ZyYWdtZW50cyI6IHRydWUsCiAgICAgICAgImFsbF9yZXF1aXJlbWVudHNfcHJlc2VydmVkIjogdHJ1ZSwKICAgICAgICAicXVhbnRpZmllcl9wcmVzZXJ2ZWQiOiB0cnVlLAogICAgICAgICJlbnRpdHlfbWV0cmljX2JpbmRpbmdfY29ycmVjdCI6IHRydWUsCiAgICAgICAgInRpbWVfY29uZGl0aW9uX3Njb3BlX2NvcnJlY3QiOiB0cnVlCiAgICAgIH0sCiAgICAgICJub3RlcyI6ICJNZXRyaWMgY29udGFpbmVkIDI0MCBkb2N1bWVudHMgZ2nhu68gcXVhbiBo4buHIGNo4bupYSB2w6AgY291bnQgb2JqZWN0LCBraMO0bmcgY8OybiBjb250YWluZWQgxJHGoW4gbOG6uy4gUmVxdWlyZW1lbnQgMjQwIGRvY3VtZW50cyDEkeG6p3kgxJHhu6cuIgogICAgfSwKICAgIHsKICAgICAgImV4YW1wbGVfaWQiOiAiZGVjb21wb3NlOjAwOSIsCiAgICAgICJjbGFpbSI6ICJPbiB0aGUgdGVzdGVkIGRhdGFzZXQsIHRoZSBtZXRob2QgaW1wcm92ZWQgcmVjYWxsLiIsCiAgICAgICJqdWRnbWVudHMiOiB7CiAgICAgICAgImZhY3R1YWxfcHJlZGljYXRlc19ub3RfZnJhZ21lbnRzIjogdHJ1ZSwKICAgICAgICAiYWxsX3JlcXVpcmVtZW50c19wcmVzZXJ2ZWQiOiB0cnVlLAogICAgICAgICJxdWFudGlmaWVyX3ByZXNlcnZlZCI6IHRydWUsCiAgICAgICAgImVudGl0eV9tZXRyaWNfYmluZGluZ19jb3JyZWN0IjogdHJ1ZSwKICAgICAgICAidGltZV9jb25kaXRpb25fc2NvcGVfY29ycmVjdCI6IHRydWUKICAgICAgfSwKICAgICAgIm5vdGVzIjogIkVudGl0eSBtZXRob2QsIG1ldHJpYyByZWNhbGw7IE9uIHRoZSB0ZXN0ZWQgZGF0YXNldCBu4bqxbSDhu58gY29uZGl0aW9uLCB0aW1lIHLhu5duZy4gS2jDtG5nIG3hu58gcuG7mW5nIGvhur90IGx14bqtbiBzYW5nIGRhdGFzZXQga2jDoWMuIgogICAgfSwKICAgIHsKICAgICAgImV4YW1wbGVfaWQiOiAiZGVjb21wb3NlOjAxMCIsCiAgICAgICJjbGFpbSI6ICJFdmVyeSB2YWxpZGF0aW9uIHNoYXJkIGltcHJvdmVkIHJlY2FsbCBhZnRlciB0aGUgcmV2aXNpb24uIiwKICAgICAgImp1ZGdtZW50cyI6IHsKICAgICAgICAiZmFjdHVhbF9wcmVkaWNhdGVzX25vdF9mcmFnbWVudHMiOiB0cnVlLAogICAgICAgICJhbGxfcmVxdWlyZW1lbnRzX3ByZXNlcnZlZCI6IHRydWUsCiAgICAgICAgInF1YW50aWZpZXJfcHJlc2VydmVkIjogdHJ1ZSwKICAgICAgICAiZW50aXR5X21ldHJpY19iaW5kaW5nX2NvcnJlY3QiOiB0cnVlLAogICAgICAgICJ0aW1lX2NvbmRpdGlvbl9zY29wZV9jb3JyZWN0IjogdHJ1ZQogICAgICB9LAogICAgICAibm90ZXMiOiAiRXZlcnkgw6FuaCB44bqhIGFsbCB0csOqbiB2YWxpZGF0aW9uIHNoYXJkOyBtZXRyaWMgaGnhu4duIGzDoCByZWNhbGwsIGtow7RuZyBjw7JuIGltcHJvdmVkIHJlY2FsbC4gYWZ0ZXIgdGhlIHJldmlzaW9uIG7hurFtIMSRw7puZyB0aW1lOyBoxrDhu5tuZyBpbXByb3ZlZCB24bqrbiBnaeG7ryBuZ3V5w6puIHRyb25nIHF1b3RlLiIKICAgIH0sCiAgICB7CiAgICAgICJleGFtcGxlX2lkIjogImRlY29tcG9zZTowMTEiLAogICAgICAiY2xhaW0iOiAiQXQgbGVhc3Qgb25lIGF1ZGl0ZWQgY2FjaGUgbm9kZSByZWR1Y2VkIHJlc3BvbnNlIHRpbWUgYWZ0ZXIgdGhlIHBhdGNoLiIsCiAgICAgICJqdWRnbWVudHMiOiB7CiAgICAgICAgImZhY3R1YWxfcHJlZGljYXRlc19ub3RfZnJhZ21lbnRzIjogdHJ1ZSwKICAgICAgICAiYWxsX3JlcXVpcmVtZW50c19wcmVzZXJ2ZWQiOiB0cnVlLAogICAgICAgICJxdWFudGlmaWVyX3ByZXNlcnZlZCI6IHRydWUsCiAgICAgICAgImVudGl0eV9tZXRyaWNfYmluZGluZ19jb3JyZWN0IjogdHJ1ZSwKICAgICAgICAidGltZV9jb25kaXRpb25fc2NvcGVfY29ycmVjdCI6IHRydWUKICAgICAgfSwKICAgICAgIm5vdGVzIjogIkF0IGxlYXN0IG9uZSDDoW5oIHjhuqEgc29tZTsgcG9wdWxhdGlvbiBhdWRpdGVkIGNhY2hlIG5vZGUgZ2nhu68gYXVkaXRlZC4gTWV0cmljIHJlc3BvbnNlIHRpbWUgZ2nhu68gY+G6oyBj4bulbTsgYWZ0ZXIgdGhlIHBhdGNoIGzDoCB0aW1lIGPhu6dhIHByZWRpY2F0ZSByZWR1Y2VkLiIKICAgIH0sCiAgICB7CiAgICAgICJleGFtcGxlX2lkIjogImRlY29tcG9zZTowMTIiLAogICAgICAiY2xhaW0iOiAiTm9uZSBvZiB0aGUgbW9uaXRvcmVkIHB1bXBzIGxlYWtlZCBkdXJpbmcgdGhlIG92ZXJuaWdodCB0ZXN0LiIsCiAgICAgICJqdWRnbWVudHMiOiB7CiAgICAgICAgImZhY3R1YWxfcHJlZGljYXRlc19ub3RfZnJhZ21lbnRzIjogdHJ1ZSwKICAgICAgICAiYWxsX3JlcXVpcmVtZW50c19wcmVzZXJ2ZWQiOiB0cnVlLAogICAgICAgICJxdWFudGlmaWVyX3ByZXNlcnZlZCI6IHRydWUsCiAgICAgICAgImVudGl0eV9tZXRyaWNfYmluZGluZ19jb3JyZWN0IjogdHJ1ZSwKICAgICAgICAidGltZV9jb25kaXRpb25fc2NvcGVfY29ycmVjdCI6IHRydWUKICAgICAgfSwKICAgICAgIm5vdGVzIjogIk5vbmUgw6FuaCB44bqhIG5vbmUgdHLDqm4gbW9uaXRvcmVkIHB1bXBzOyBtZXRyaWMgbGVha2VkIGdp4buvIGThuqFuZyBkxrDGoW5nIMSR4buDIHF1YW50aWZpZXIgbm9uZSBtYW5nIHBo4bunIMSR4buLbmguIGR1cmluZyB0aGUgb3Zlcm5pZ2h0IHRlc3QgdsOgIG1vZGlmaWVyIG1vbml0b3JlZCDEkcaw4bujYyBnaeG7ry4iCiAgICB9LAogICAgewogICAgICAiZXhhbXBsZV9pZCI6ICJkZWNvbXBvc2U6MDEzIiwKICAgICAgImNsYWltIjogIlRoZSBOb3J0aCBnYXRld2F5IHJlZHVjZWQgaXRzIGVycm9yIHJhdGUgYWZ0ZXIgY2FsaWJyYXRpb24uIiwKICAgICAgImp1ZGdtZW50cyI6IHsKICAgICAgICAiZmFjdHVhbF9wcmVkaWNhdGVzX25vdF9mcmFnbWVudHMiOiB0cnVlLAogICAgICAgICJhbGxfcmVxdWlyZW1lbnRzX3ByZXNlcnZlZCI6IHRydWUsCiAgICAgICAgInF1YW50aWZpZXJfcHJlc2VydmVkIjogdHJ1ZSwKICAgICAgICAiZW50aXR5X21ldHJpY19iaW5kaW5nX2NvcnJlY3QiOiB0cnVlLAogICAgICAgICJ0aW1lX2NvbmRpdGlvbl9zY29wZV9jb3JyZWN0IjogdHJ1ZQogICAgICB9LAogICAgICAibm90ZXMiOiAiRW50aXR5IE5vcnRoIGdhdGV3YXkgZ2nhu68gxJHhu4tuaCBkYW5oIE5vcnRoOyBtZXRyaWMgZXJyb3IgcmF0ZSBraMO0bmcga8OobSByZWR1Y2VkL2l0cy4gUXVhbiBo4buHIHJlZHVjZWQgdsOgIHRpbWUgYWZ0ZXIgY2FsaWJyYXRpb24gduG6q24gbmd1ecOqbiB0cm9uZyBxdW90ZS4iCiAgICB9LAogICAgewogICAgICAiZXhhbXBsZV9pZCI6ICJkZWNvbXBvc2U6MDE0IiwKICAgICAgImNsYWltIjogIkV2ZXJ5IHdlZWtseSB0cmlhbCBjb25kdWN0ZWQgaW4gTWF5IGltcHJvdmVkIHRocm91Z2hwdXQuIiwKICAgICAgImp1ZGdtZW50cyI6IHsKICAgICAgICAiZmFjdHVhbF9wcmVkaWNhdGVzX25vdF9mcmFnbWVudHMiOiB0cnVlLAogICAgICAgICJhbGxfcmVxdWlyZW1lbnRzX3ByZXNlcnZlZCI6IHRydWUsCiAgICAgICAgInF1YW50aWZpZXJfcHJlc2VydmVkIjogdHJ1ZSwKICAgICAgICAiZW50aXR5X21ldHJpY19iaW5kaW5nX2NvcnJlY3QiOiB0cnVlLAogICAgICAgICJ0aW1lX2NvbmRpdGlvbl9zY29wZV9jb3JyZWN0IjogdHJ1ZQogICAgICB9LAogICAgICAibm90ZXMiOiAiRXZlcnkgw6FuaCB44bqhIGFsbCB0csOqbiB3ZWVrbHkgdHJpYWw7IG1ldHJpYyB0aHJvdWdocHV0IGtow7RuZyBjw7JuIGltcHJvdmVkIHRocm91Z2hwdXQuIGNvbmRpdGlvbiBpbiBNYXkgbOG7jWMgY8OhYyB0cmlhbCBjb25kdWN0ZWQgaW4gTWF5LCBraMO0bmcga2jhurNuZyDEkeG7i25oIHZp4buHYyBj4bqjaSB0aGnhu4duIGRp4buFbiByYSB0cm9uZyBNYXkuIFF1b3RlIMSR4bqneSDEkeG7pyBnaeG7ryBhdHRhY2htZW50IGNvbmR1Y3RlZCBpbiBNYXk7IHNjaGVtYSBzY29wZSBjaMawYSBjw7MgdHLGsOG7nW5nIHJlc3RyaWN0aW9uLW93bmVyIHJpw6puZy4iCiAgICB9LAogICAgewogICAgICAiZXhhbXBsZV9pZCI6ICJkZWNvbXBvc2U6MDE1IiwKICAgICAgImNsYWltIjogIlRoZSBwYXRjaCByZWR1Y2VkIG1lZGlhbiByZXF1ZXN0IGxhdGVuY3kuIiwKICAgICAgImp1ZGdtZW50cyI6IHsKICAgICAgICAiZmFjdHVhbF9wcmVkaWNhdGVzX25vdF9mcmFnbWVudHMiOiB0cnVlLAogICAgICAgICJhbGxfcmVxdWlyZW1lbnRzX3ByZXNlcnZlZCI6IHRydWUsCiAgICAgICAgInF1YW50aWZpZXJfcHJlc2VydmVkIjogdHJ1ZSwKICAgICAgICAiZW50aXR5X21ldHJpY19iaW5kaW5nX2NvcnJlY3QiOiB0cnVlLAogICAgICAgICJ0aW1lX2NvbmRpdGlvbl9zY29wZV9jb3JyZWN0IjogdHJ1ZQogICAgICB9LAogICAgICAibm90ZXMiOiAiTWV0cmljIG1lZGlhbiByZXF1ZXN0IGxhdGVuY3kgZ2nhu68gY+G6oyBtZWRpYW4gdsOgIHJlcXVlc3Q7IGtow7RuZyByw7p0IGPDsm4gbGF0ZW5jeS4gRW50aXR5IHBhdGNoIHbDoCBoxrDhu5tuZyByZWR1Y2VkIHRyb25nIHF1b3RlIMSRw7puZy4iCiAgICB9LAogICAgewogICAgICAiZXhhbXBsZV9pZCI6ICJkZWNvbXBvc2U6MDE2IiwKICAgICAgImNsYWltIjogIlRoZSBjb25maWd1cmF0aW9uIGNoYW5nZSBpbmNyZWFzZWQgcHJlY2lzaW9uLiIsCiAgICAgICJqdWRnbWVudHMiOiB7CiAgICAgICAgImZhY3R1YWxfcHJlZGljYXRlc19ub3RfZnJhZ21lbnRzIjogdHJ1ZSwKICAgICAgICAiYWxsX3JlcXVpcmVtZW50c19wcmVzZXJ2ZWQiOiB0cnVlLAogICAgICAgICJxdWFudGlmaWVyX3ByZXNlcnZlZCI6IHRydWUsCiAgICAgICAgImVudGl0eV9tZXRyaWNfYmluZGluZ19jb3JyZWN0IjogdHJ1ZSwKICAgICAgICAidGltZV9jb25kaXRpb25fc2NvcGVfY29ycmVjdCI6IHRydWUKICAgICAgfSwKICAgICAgIm5vdGVzIjogIkVudGl0eSBjb25maWd1cmF0aW9uIGNoYW5nZSDEkeG6p3kgxJHhu6c7IG1ldHJpYyBwcmVjaXNpb24ga2jDtG5nIGzhuqtuIHbhu5tpIGNoYW5nZSB0cm9uZyBzdWJqZWN0LiBIxrDhu5tuZyBpbmNyZWFzZWQgZ2nhu68gdHJvbmcgcXVvdGUuIgogICAgfSwKICAgIHsKICAgICAgImV4YW1wbGVfaWQiOiAiZGVjb21wb3NlOjAxNyIsCiAgICAgICJjbGFpbSI6ICJBZnRlciB0dW5pbmcsIHRoZSBleHRyYWN0b3IgZGVjcmVhc2VkIG1lbW9yeSB1c2UgYW5kIGluY3JlYXNlZCB0b2tlbiByZWNhbGwuIiwKICAgICAgImp1ZGdtZW50cyI6IHsKICAgICAgICAiZmFjdHVhbF9wcmVkaWNhdGVzX25vdF9mcmFnbWVudHMiOiB0cnVlLAogICAgICAgICJhbGxfcmVxdWlyZW1lbnRzX3ByZXNlcnZlZCI6IHRydWUsCiAgICAgICAgInF1YW50aWZpZXJfcHJlc2VydmVkIjogdHJ1ZSwKICAgICAgICAiZW50aXR5X21ldHJpY19iaW5kaW5nX2NvcnJlY3QiOiB0cnVlLAogICAgICAgICJ0aW1lX2NvbmRpdGlvbl9zY29wZV9jb3JyZWN0IjogdHJ1ZQogICAgICB9LAogICAgICAibm90ZXMiOiAiSGFpIHByZWRpY2F0ZSBkZWNyZWFzZWQgbWVtb3J5IHVzZSB2w6AgaW5jcmVhc2VkIHRva2VuIHJlY2FsbCB0w6FjaCDEkcO6bmc7IHBhcnQgc2F1IGvhur8gdGjhu6thIGV4dHJhY3Rvci4gQWZ0ZXIgdHVuaW5nIGzDoCBwcmVmaXggY2h1bmcgY8OzIGThuqV1IHBo4bqpeSwgbsOqbiB0aW1lIGPhu6dhIGPhuqMgaGFpIHBhcnQgxJHDum5nLiBLaMO0bmcgdHLhu5luIGhhaSBoxrDhu5tuZyB0aGF5IMSR4buVaSBob+G6t2MgaGFpIG1ldHJpYy4iCiAgICB9LAogICAgewogICAgICAiZXhhbXBsZV9pZCI6ICJkZWNvbXBvc2U6MDE4IiwKICAgICAgImNsYWltIjogIkV2ZXJ5IGJhdGNoIGluIHRoZSBsb3ctdGVtcGVyYXR1cmUgZ3JvdXAgZ2FpbmVkIHlpZWxkIGFmdGVyIHRoZSB1cGRhdGUuIiwKICAgICAgImp1ZGdtZW50cyI6IHsKICAgICAgICAiZmFjdHVhbF9wcmVkaWNhdGVzX25vdF9mcmFnbWVudHMiOiB0cnVlLAogICAgICAgICJhbGxfcmVxdWlyZW1lbnRzX3ByZXNlcnZlZCI6IHRydWUsCiAgICAgICAgInF1YW50aWZpZXJfcHJlc2VydmVkIjogdHJ1ZSwKICAgICAgICAiZW50aXR5X21ldHJpY19iaW5kaW5nX2NvcnJlY3QiOiB0cnVlLAogICAgICAgICJ0aW1lX2NvbmRpdGlvbl9zY29wZV9jb3JyZWN0IjogdHJ1ZQogICAgICB9LAogICAgICAibm90ZXMiOiAiRXZlcnkgw6FuaCB44bqhIGFsbDsgZW50aXR5IGJhdGNoIGvhur90IGjhu6NwIGNvbmRpdGlvbiBpbiB0aGUgbG93LXRlbXBlcmF0dXJlIGdyb3VwIGdp4buvIMSRw7puZyBwb3B1bGF0aW9uLiBNZXRyaWMgeWllbGQga2jDtG5nIGPDsm4gZ2FpbmVkIHlpZWxkOyBhZnRlciB0aGUgdXBkYXRlIGzDoCB0aW1lLCBnYWluZWQgZ2nhu68gbmd1ecOqbiB0cm9uZyBxdW90ZS4iCiAgICB9CiAgXQp9Cg=='
review_sha256 = 'b1caf56e66a869d7e9c9a3d3b8443449ee3915e1a1a1991304031704364104dc'
try:
    probe_data=base64.b64decode(probe_zip_b64,validate=True)
    assert hashlib.sha256(probe_data).hexdigest()==probe_zip_sha256
    (out/'probe-original.zip').write_bytes(probe_data)
    original_probe=run_root/'probe-original'
    original_probe.mkdir(exist_ok=False)
    with zipfile.ZipFile(io.BytesIO(probe_data)) as archive:
        assert archive.testzip() is None
        records=json.loads(archive.read('output-manifest.json'))['files']
        names=[r['path'] for r in records]
        assert len(names)==len(set(names)) and set(archive.namelist())==set(names)|{'output-manifest.json'}
        for item in records:
            relative=PurePosixPath(item['path'])
            assert not relative.is_absolute() and '..' not in relative.parts and '\\' not in item['path'] and ':' not in item['path']
            data=archive.read(item['path'])
            assert len(data)==item['size'] and hashlib.sha256(data).hexdigest()==item['sha256']
            target=original_probe/relative
            target.parent.mkdir(parents=True,exist_ok=True)
            target.write_bytes(data)
    reviewed_probe=out/'reviewed-probe'
    shutil.copytree(original_probe/'decomposition-probe',reviewed_probe)
    review_data=base64.b64decode(review_b64,validate=True)
    assert hashlib.sha256(review_data).hexdigest()==review_sha256
    (reviewed_probe/'semantic-review.json').write_bytes(review_data)
    provenance=json.loads((out/'notebook-provenance.json').read_text())
    provenance.update(purpose='probe-admitted full39',original_probe_run='20261017T173345Z-905edcb5',
        original_probe_zip_sha256=probe_zip_sha256,semantic_review_sha256=review_sha256)
    (out/'notebook-provenance.json').write_text(json.dumps(provenance,indent=2))
    bundle()
except BaseException:
    download_archive()
    raise
print('Original probe preserved. Human review is installed only in a separate reviewed copy.')


**2 — Môi trường T4**

In [ ]:
try:
    run('gpu-host',['nvidia-smi'])
    run('venv', [sys.executable, '-m', 'venv', '--without-pip', str(venv)])
    run('bootstrap', [sys.executable, '-m', 'pip', '--python', python, 'install', '--no-cache-dir', 'pip==26.2.1', 'wrapt==2.1.2'])
    run('torch', [python, '-m', 'pip', 'install', '--no-cache-dir', 'torch==2.10.0', 'torchvision==0.25.0', '--index-url', 'https://download.pytorch.org/whl/cu128'])
    run('runtime', [python, '-m', 'pip', 'install', '--no-cache-dir', '-r', 'requirements-orchestrator.txt', '-r', 'requirements-atomic-verifier.txt'])
    run('pip-check', [python, '-m', 'pip', 'check'])
    run('pip-freeze', [python, '-m', 'pip', 'freeze'])
except BaseException:
    download_archive()
    raise


**3 — Admission và validation**

In [ ]:
try:
    admission_code="import json; from pathlib import Path; from research_rag.evaluation.evaluate_atomic_verifier import require_probe_review; print(json.dumps(require_probe_review(Path("+repr(str(reviewed_probe))+")),ensure_ascii=False))"
    run('probe-admission',[python,'-c',admission_code],timeout=300)
except BaseException:
    download_archive()
    raise
print('Human admission passed before model-cache downloads or inference.')
try:
    run('regression',[python,'-m','unittest','research_rag.tests.test_atomic_verifier','research_rag.tests.test_atomic_experiment','research_rag.tests.test_atomic_decoding','research_rag.tests.test_atomic_decomposition_v5','research_rag.tests.test_atomic_contract_v5','research_rag.tests.test_summary_schema','research_rag.tests.test_summary_trace','research_rag.tests.test_atomic_metric_v6','research_rag.tests.test_atomic_metric_v7','research_rag.tests.test_atomic_assessment_v8','research_rag.tests.test_atomic_smoke_cellfix','research_rag.tests.test_atomic_smoke_cellfix2','research_rag.tests.test_atomic_smoke_cellfix3','research_rag.tests.test_atomic_smoke_cellfix4','research_rag.tests.test_atomic_metric_cellfix5','-v'],timeout=900)
    run('cuda-preflight',[python,'-c',"import json,torch; assert torch.cuda.is_available(); print(json.dumps({'torch':torch.__version__,'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0)}))"])
    for name,flag in (('probe-validate-only','--decomposition-probe'),('smoke-validate-only','--assessment-smoke'),('verifier-validate-only',None)):
        command=[python,'-m','research_rag.evaluation.evaluate_atomic_verifier','--output-dir',str(out/'unused'),'--validate-only']
        if flag: command.append(flag)
        run(name,command)
    run('pinned-model-cache',[python,'-c',"from huggingface_hub import snapshot_download; print(snapshot_download('Qwen/Qwen2.5-3B-Instruct',revision='aa8e72537993ba99e69dfaafa59ed015b17504d1',ignore_patterns=['*.bin','*.gguf','original/*']))"],timeout=1800)
except BaseException:
    download_archive()
    raise
print('Regression, CUDA, freeze and pinned model-cache checks passed.')


**4 — Full39**

In [ ]:
full39_passed=False
try:
    required=('gpu-host','venv','bootstrap','torch','runtime','pip-check','pip-freeze','regression',
        'cuda-preflight','probe-validate-only','smoke-validate-only','verifier-validate-only','pinned-model-cache','probe-admission')
    assert all(statuses.get(n,{}).get('exit_code')==0 for n in required),'Complete cells 1-3 including human admission.'
    assert not (out/'verifier39').exists(),'Run already has verifier39 output; preserve it and use a fresh runtime for another run.'
    code=run('verifier39',[python,'-u','-m','research_rag.evaluation.evaluate_atomic_verifier',
        '--output-dir',str(out/'verifier39'),'--probe-dir',str(reviewed_probe),'--online-model-downloads'],
        timeout=5520,required=False)
    for name in ('summary.json','metrics.json'):
        path=out/'verifier39'/name
        print(name,path.read_text() if path.exists() else 'Incomplete; inspect saved logs.')
    path=out/'verifier39/metrics.json'
    metrics=json.loads(path.read_text()) if path.exists() else {}
    full39_passed=(code==0 and metrics.get('automated_gate_pass') is True and metrics.get('n')==39
        and metrics.get('correct_labels')==39 and metrics.get('false_support')==0 and metrics.get('runtime_failed_rows')==0)
finally:
    download_archive()
print('FULL39 AUTOMATED GATE:',full39_passed)
print('Keep the ZIP including failures. Evidence review and E2E are still required; this does not promote serving.')
